In [ ]:
# Cell 1 - load the API key
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check .env at the repo root"

In [ ]:
# Cell 2 - fixed paths, independent of where the kernel starts
from pathlib import Path

PROJECT_DIR = Path(find_dotenv(usecwd=True)).parent / "Code/projects/rbi-fsr-rag"
PDF_DIR = PROJECT_DIR / "pdfs"
TEXT_DIR = PROJECT_DIR / "pdfs_text"

assert (PROJECT_DIR / "rbi_fsr.csv").exists(), PROJECT_DIR

In [ ]:
# Cell 3 - read the list of reports
import csv

with open(PROJECT_DIR / "rbi_fsr.csv", newline="", encoding="utf-8") as f:
    reports = list(csv.DictReader(f))

for r in reports:
    print(r["edition"], r["release_date"], r["title"])

In [ ]:
# Cell 4 - download one PDF (first report) into memory
import requests

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/pdf",
}

r = reports[0]
response = requests.get(r["pdf_url"], headers=HEADERS, timeout=60)

print(response.status_code, response.headers.get("Content-Type"), len(response.content) / 1e6, "MB")

In [ ]:
# Cell 5 - check the bytes are a real PDF, not an HTML challenge page
print(response.content[:5])
assert response.content.startswith(b"%PDF-"), "Not a PDF - got an HTML challenge page?"

In [ ]:
# Cell 6 - download all reports to pdfs/, skipping ones already saved
PDF_DIR.mkdir(exist_ok=True)

for r in reports:
    path = PDF_DIR / f"{r['edition']}.pdf"

    if path.exists() and path.read_bytes()[:5] == b"%PDF-":
        print("skip", path.name)
        continue

    response = requests.get(r["pdf_url"], headers=HEADERS, timeout=60)
    response.raise_for_status()
    assert response.content.startswith(b"%PDF-"), f"Not a PDF: {r['edition']}"

    path.write_bytes(response.content)
    print("saved", path.name, round(len(response.content) / 1e6, 1), "MB")

In [ ]:
# Cell 7 - convert ONE PDF to Markdown and look at it
from markitdown import MarkItDown

md = MarkItDown()

result = md.convert(PDF_DIR / "2026-H1.pdf")
text = result.text_content

print(len(text), "characters")
print(text[:1500])

In [ ]:
# Cell 8 - convert all PDFs and save as pdfs_text/<edition>.md, skipping ones already converted
TEXT_DIR.mkdir(exist_ok=True)

for r in reports:
    pdf_path = PDF_DIR / f"{r['edition']}.pdf"
    md_path = TEXT_DIR / f"{r['edition']}.md"

    if md_path.exists() and md_path.stat().st_size > 0:
        print("skip", md_path.name)
        continue

    text = md.convert(pdf_path).text_content
    md_path.write_text(text, encoding="utf-8")
    print("saved", md_path.name, len(text), "characters")

**Q1 result:** markitdown converted all 4 reports to Markdown.

| edition | characters |
|---|---|
| 2026-H1 | 880,538 |
| 2025-H2 | 735,828 |
| 2025-H1 | 635,073 |
| 2024-H2 | 491,497 |

Noted, not cleaned (we follow the homework): chart axis numbers end up as loose lines, some tables come out as wide sparse Markdown tables, two-column pages sometimes interleave, and page breaks are lost (no page numbers).

In [ ]:
# Cell 9 - read one Markdown file, keep only non-empty lines
lines = (TEXT_DIR / "2026-H1.md").read_text(encoding="utf-8").splitlines()
non_empty = [line for line in lines if line.strip() != ""]

print(len(lines), "lines ->", len(non_empty), "non-empty")
print(non_empty[:10])

In [ ]:
# Cell 10 - turn each report into {"source": filename, "content": [non-empty lines]}
documents_raw = []

for md_path in sorted(TEXT_DIR.glob("*.md")):
    lines = md_path.read_text(encoding="utf-8").splitlines()
    non_empty = [line for line in lines if line.strip() != ""]
    documents_raw.append({"source": md_path.name, "content": non_empty})

for doc in documents_raw:
    print(doc["source"], len(doc["content"]), "lines")

In [ ]:
# Cell 11 - chunk with a sliding window: 100 lines per chunk, move forward 50 lines each time
from gitsource import chunk_documents

chunks = chunk_documents(documents_raw, size=100, step=50)

print(len(chunks), "chunks")
print(chunks[0].keys())

In [ ]:
# Cell 12 - chunks per report, total, and a look at one chunk
from collections import Counter

per_report = Counter(chunk["source"] for chunk in chunks)
for source, count in sorted(per_report.items()):
    print(source, count)
print("total", len(chunks))

sample = chunks[1]
print(sample["source"], "start =", sample["start"], "lines =", len(sample["content"]))
print("\n".join(sample["content"][:5]))

**Q2 result:** size=100, step=50 -> 2024-H2 = 140, 2025-H1 = 195, 2025-H2 = 215, 2026-H1 = 251, **total = 801 chunks**.

In [ ]:
# Cell 13 - copy each chunk with its list of lines joined into one string (chunks stays untouched)
documents = []

for chunk in chunks:
    doc = chunk.copy()
    doc["content"] = "\n".join(chunk["content"])
    documents.append(doc)

print(len(documents), "documents")
print(type(chunks[0]["content"]), "->", type(documents[0]["content"]))

In [ ]:
# Cell 14 - build the keyword index: search in "content", filter by "source"
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["source"])
index.fit(documents)

print(len(index.docs), "documents indexed")
assert len(index.docs) == len(chunks), "index size doesn't match the chunk count"

**Q3 result:** **801 documents indexed** (equals the chunk count).

In [ ]:
# Cell 15 - the one fixed question for Steps 5-7, and a search over the index
QUESTION = "gross NPA ratio of scheduled commercial banks"

results = index.search(QUESTION, num_results=5)

for i, res in enumerate(results, 1):
    print(i, res["source"], "start =", res["start"])

In [ ]:
# Cell 16 - read the top result: is it really about gross NPAs?
top = results[0]
print(top["source"], "start =", top["start"])
print()
print(top["content"][:2000])

**Q4 result:** for "gross NPA ratio of scheduled commercial banks", the top result came from **2025-H1.md (FSR June 2025)**, start = 4300. It is relevant: "SCBs continued to record improvement in their asset quality, with the GNPA ratio and NNPA ratio declining to multi-decadal lows of 2.3 per cent and 0.5 per cent".

In [ ]:
# Cell 17 - the homework's RAG code, with only the instructions reworded for the RBI reports
import json
from openai import OpenAI

openai_client = OpenAI()

instructions = """
You're an assistant for the RBI Financial Stability Reports, your task is to answer the QUESTION
from the users using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

answer = rag(QUESTION)
print(answer)

In [ ]:
# Cell 18 - same RAG, but llm and rag now also return the token usage
def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text, response.usage

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm(prompt, instructions)
    return answer, usage

answer, usage = rag(QUESTION)
print(answer)
print()
print("input tokens:", usage.input_tokens)
print("output tokens:", usage.output_tokens)

**Q5 result:** one RAG query with `gpt-4o-mini` used **9,750 input tokens** and 89 output tokens. The answer: GNPA ratio of SCBs at 2.3% and NNPA at 0.5% as of March 2025 (see the Finding below on why it isn't March 2026).

In [ ]:
# Cell 19 - the homework's response model: the shape we want the answer in
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [ ]:
# Cell 20 - same RAG, but the model must answer in the RAGResponse shape
from IPython.display import Markdown, display

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return response.output_parsed, response.usage

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm(prompt, instructions)
    return answer, usage

structured_answer, structured_usage = rag(QUESTION)

r = structured_answer
print(f"Answer type:  {r.answer_type}")
print(f"Found answer: {r.found_answer}")
print(f"Confidence:   {r.confidence:.2f} - {r.confidence_explanation}")

display(Markdown(r.answer))

print("Follow-up questions:")
for q in r.followup_questions:
    print(f"  - {q}")

In [ ]:
# Cell 21 - compare input tokens: plain text (Cell 18) vs structured (Cell 20)
plain = usage.input_tokens
structured = structured_usage.input_tokens

print(f"{'':<14}{'input tokens':>14}")
print(f"{'unstructured':<14}{plain:>14,}")
print(f"{'structured':<14}{structured:>14,}")
print(f"{'difference':<14}{structured - plain:>+14,}")

**Q6 result:** structured output used 9,936 input tokens vs 9,750 unstructured, so **+186 more input tokens**. That's the JSON schema of `RAGResponse` (field names, types, the 5 `answer_type` options, descriptions) sent with the request.

**Finding: the answer used March 2025 data, not the latest (March 2026).**

The RAG answer quoted the GNPA ratio for March 2025, although the June 2026 FSR has the March 2026 figure ("...March 2026, with GNPA ratio declining to a multi-..."). These are the top 5 search results for `QUESTION`:

| rank | report | "GNPA" in chunk | dates in chunk |
|---|---|---|---|
| 1 | 2025-H1 (June 2025) | 11 | March 2025, September 2024 |
| 2 | 2024-H2 (Dec 2024) | 1 | - |
| 3 | 2026-H1 (June 2026) | 1 | March 2026 |
| 4 | 2025-H1 (June 2025) | 0 | March 2025 |
| 5 | 2025-H2 (Dec 2025) | 1 | end-September 2025 |

Why:
1. **Keyword search has no notion of time.** minsearch ranks by word overlap, so the June 2025 chunk wins with 11 mentions of "GNPA".
2. **The question doesn't say "latest".** The same question is equally valid for every report.
3. **The prompt doesn't tell the model which report is newest.** Chunks only carry `source` (e.g. `2025-H1.md`), and the instructions say nothing about preferring recent data.

Steps 5-7 are kept exactly as the homework wrote them, so the Q4-Q6 numbers stay reproducible. Fix, as an extension: add `edition` / `release_date` to each chunk, tell the model to prefer the most recent report and name the report it quotes, and/or filter with `index.search(..., filter_dict={"source": "2026-H1.md"})`.

## Step 9 - Extension: prefer the latest report

Fix for the Finding above, in new cells (Steps 5-7 stay as the homework wrote them):
- **Fix 1:** same search, but the instructions explain what `source` means and tell the model to use the most recent report and name it.
- **Fix 2:** Fix 1 plus searching each report separately (top 2 per report), so the latest report is always in the context.

In [ ]:
# Cell 22 - fix 1: same search, new instructions that explain "source" and ask for the latest report
instructions_latest = """
You're an assistant for the RBI Financial Stability Reports. Answer the QUESTION using the CONTEXT.
Each context item has a "source" like "2026-H1.md": H1 = June report (data as of end-March),
H2 = December report (data as of end-September). When reports give different figures,
use the most recent report and say which report and date the figure comes from.
"""

def rag_latest(query, search_fn):
    search_results = search_fn(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions_latest)

fix1_answer, fix1_usage = rag_latest(QUESTION, search)

display(Markdown(fix1_answer.answer))
print("input tokens:", fix1_usage.input_tokens)

In [ ]:
# Cell 23 - fix 2: top 2 chunks from EACH report, so every edition is always in the context
SOURCES = sorted(doc["source"] for doc in documents_raw)

def search_per_report(question, per_report=2):
    results = []
    for source in SOURCES:
        results += index.search(question, filter_dict={"source": source}, num_results=per_report)
    return results

for res in search_per_report(QUESTION):
    print(res["source"], "start =", res["start"])

In [ ]:
# Cell 24 - run fix 2: per-report search + the new instructions
fix2_answer, fix2_usage = rag_latest(QUESTION, search_per_report)

display(Markdown(fix2_answer.answer))
print("input tokens:", fix2_usage.input_tokens)

In [ ]:
# Cell 25 - compare the three versions: what each costs in input tokens
rows = [
    ("homework (Cell 20)", structured_usage.input_tokens),
    ("fix 1 (Cell 22)", fix1_usage.input_tokens),
    ("fix 2 (Cell 24)", fix2_usage.input_tokens),
]

print(f"{'':<20}{'input tokens':>14}")
for name, tokens in rows:
    print(f"{name:<20}{tokens:>14,}")

**Step 9 result: neither fix got the March 2026 figure. The problem is retrieval, not the prompt.**

| version | answer | input tokens |
|---|---|---|
| homework (Cell 20) | 2.3%, March 2025 | 9,936 |
| fix 1 - new instructions (Cell 22) | 2.3%, March 2025, *now names the 2025-H1 report* | 9,991 |
| fix 2 - top 2 per report (Cell 24) | 2.6%, September 2024 | 14,850 |

The correct answer is in the June 2026 FSR, at line 5447 of `2026-H1.md`: *"Asset quality of SCBs improved further in March 2026, with GNPA ratio declining to a multi-decadal low level of **1.8 per cent**."* It sits in the chunk starting at line 5400, which **no search returned**:
- The 2026-H1 chunks that were retrieved (start 5200, 3450) only have a scrambled overview table (`GNPA Ratio | 7.3 3.9 | 2.8 2.3 | 1.8`), so the model can't tell which number is which year.
- The question's wording doesn't match the report's: the report says "GNPA" and "SCBs", not "gross NPA" and "scheduled commercial banks". Even with the report's own words ("GNPA ratio of SCBs"), the 1.8% chunk isn't in the top 5 for 2026-H1.
- markitdown interleaves the two-column pages ("...declining to a multi- 2.8 The provisioning coverage ratio..."), which dilutes the keyword match.

Lessons:
1. The instructions change helped one thing: the answer now **names its report**, so the wrong year is visible instead of hidden.
2. Per-report search costs about 50% more tokens and didn't help, because the right chunk was never among each report's top 2.
3. The real fixes are on the retrieval side: smaller chunks, column-aware / page-aware extraction (also needed for page citations), and vector or hybrid search, which matches meaning, not just words.

## Tough questions - stress-testing the RAG

5 hard questions, each shown with the full structured output (type, found, confidence, answer, follow-ups) plus the chunks it used and the tokens it cost. For 4 of them the correct answer was looked up in the reports by hand (`expected`), so each answer can be marked right or wrong. The last one is **not in any report**, to test whether `found_answer` is honest.

In [ ]:
# Cell 26 - ask(): one question in, everything out
def ask(question, expected=None, search_fn=search):
    search_results = search_fn(question)
    prompt = build_prompt(question, search_results)
    r, usage = llm(prompt, instructions_latest)

    display(Markdown(f"### Q: {question}"))
    print(f"Answer type:  {r.answer_type}")
    print(f"Found answer: {r.found_answer}")
    print(f"Confidence:   {r.confidence:.2f} - {r.confidence_explanation}")

    display(Markdown(r.answer))

    print("Follow-up questions:")
    for q in r.followup_questions:
        print(f"  - {q}")

    print()
    print("Chunks used: ", ", ".join(f"{res['source']}@{res['start']}" for res in search_results))
    print(f"Tokens:       {usage.input_tokens:,} in / {usage.output_tokens:,} out")
    print(f"Expected:     {expected or '(no reference answer)'}")
    print("-" * 80)

In [ ]:
# Cell 27 - five tough questions (5 gpt-4o-mini calls, about 50k input tokens in total)
TOUGH_QUESTIONS = [
    (
        "How did the GNPA ratio of SCBs change between March 2025 and March 2026?",
        "2.3% (March 2025, June 2025 FSR) -> 1.8% (March 2026, June 2026 FSR), a multi-decadal low",
    ),
    (
        "In the June 2026 stress test, what GNPA ratio does the severe scenario (Shock 2) assume, "
        "and how many banks would breach the minimum CRAR?",
        "GNPA of 46 select SCBs rises to 8.1%; four banks with 12% of SCB assets breach minimum CRAR",
    ),
    (
        "Compare the provisioning coverage ratio (PCR) of SCBs in March 2025 and March 2026.",
        "76.3% (March 2025) vs 75.6% (June 2026 FSR, 'broadly stable')",
    ),
    (
        "What are the main risks to India's financial stability in the June 2026 report?",
        None,
    ),
    (
        "What did the RBI decide on the repo rate in its October 2026 policy meeting?",
        "Not in any of the 4 reports -> found_answer should be False",
    ),
]

for question, expected in TOUGH_QUESTIONS:
    ask(question, expected)

**Tough questions result: 2 of 5 right, and confidence doesn't track correctness.**

| # | question | RAG answer | expected | confidence | verdict |
|---|---|---|---|---|---|
| 1 | GNPA, March 2025 -> March 2026 | 2.5% -> 2.3%, "per June 2026 FSR" | 2.3% -> 1.8% | 0.95 | **wrong** - both numbers shifted a year; the 1.8% chunk was not retrieved again |
| 2 | Shock 2 GNPA + banks breaching CRAR | 8.1%, four banks | 8.1% (46 select SCBs), four banks (12% of assets) | 0.90 | **right** - only 1 of 5 chunks was from 2026-H1, but it was the right one |
| 3 | PCR, March 2025 vs March 2026 | 76.3% -> 72.6% "per December 2025 FSR" | 76.3% -> 75.6% | 0.95 | **wrong** - a December report can't have March 2026 data; no 2026-H1 chunk retrieved |
| 4 | main risks, June 2026 report | 6 plausible risks, "in the June 2026 FSR" | - | 0.90 | **ungrounded** - none of the 5 chunks came from 2026-H1, so the risks are from older reports, relabelled |
| 5 | repo rate, October 2026 | "not in the context" | not in any report | 0.30 | **right** - `found_answer = False`, low confidence |

What this shows:
1. **The retrieval miss from Step 9 is systematic.** In 3 of the 4 questions about June 2026, the needed 2026-H1 chunk wasn't in the top 5.
2. **`instructions_latest` makes it worse in one way:** the model now *labels* answers with the latest report even when no chunk from it was retrieved (Q1, Q4). The report name has to come from the chunk's `source`, not from the model.
3. **The confidence score is not trustworthy.** 0.95 on both wrong numeric answers, the same as on correct ones. Only the out-of-scope question (Q5) got low confidence.
4. **What works:** refusing out-of-scope questions (Q5), and specific numbers when the right chunk *is* retrieved (Q2).

Next: fix retrieval (smaller chunks, column-aware extraction, vector/hybrid search), then re-run these 5 questions as a mini evaluation set.